#  Internshala Data Analyst Internship Scraper
## ETL Pipeline — Extract, Transform, Load

### Problem Statement
Finding relevant Data Analyst internships manually istime-consuming. This pipeline automates the collection,cleaning and storage of live internship listings from Internshala — India's largest internship platform.

### What this project does
- Extract: Scrapes live Data Analyst internship listings
- Transform: Cleans missing values and adds metadata
- Load: Stores structured data in a SQLite database

In [53]:
# IMPORTS
import requests
import csv
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import sqlite3

In [42]:
# EXTRACT PHASE
#fetching Data Analyst internship listings from Intershala
#Using browser headers to avoid being blocked by the website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

url = "https://internshala.com/internships/data-analyst-internship"

In [43]:
# Sending HTTP GET request to Internshala
response = requests.get(url, headers=headers)
print(f"Status code: {response.status_code}")

soup = BeautifulSoup(response.text, "html.parser")

Status code: 200


In [44]:
# Finding internship cards
# Each internship listing on Internshala is wrapped in a div with class "individual_internship"
jobs = soup.find_all("div", class_="individual_internship")
print(f"Found {len(jobs)} internships")

Found 41 internships


In [45]:
# Extracting key fields from each job card: Title, Company, Location, Duration, Stipend
# Using try-except to handle missing fields gracefully
with open("internshala_jobs.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Title", "Company", "Location", "Duration", "Stipend"])

    for job in jobs:
        try:
            title = job.find("h3").text.strip() if job.find("h3") else "N/A"
            company = job.find("h4").text.strip() if job.find("h4") else "N/A"
            location = job.find("a", class_="location_link")
            location = location.text.strip() if location else "Remote"
            duration = job.find("div", class_="row-1-item")
            duration = duration.text.strip() if duration else "N/A"
            stipend = job.find("span", class_="stipend")
            stipend = stipend.text.strip() if stipend else "N/A"

            writer.writerow([title, company, location, duration, stipend])
        except Exception as e:
            print(f"Skipping one entry: {e}")
            continue

print("Saved to internshala_jobs.csv")


Saved to internshala_jobs.csv


In [46]:
# Loading raw CSV into a Pandas DataFrame for cleaning and transformation
df=pd.read_csv('internshala_jobs.csv')

In [47]:
df.head(10)

,Title,Company,Location,Duration,Stipend
0,Data Entry,NaN,Remote,Gurgaon,"₹ 8,000 - 10,000 /month"
1,Support Associate,NaN,Remote,Work from home,"₹ 1,000 /month"
2,Management - Presentation Specialist,NaN,Remote,Gurgaon,"₹ 15,000 - 25,000 /month"
3,Customer Service/Customer Support,NaN,Remote,Work from home,"₹ 5,000 - 12,000 /month"
4,Client Engagement & Wealth Management Support,NaN,Remote,Mumbai,"₹ 3,000 - 7,000 /month"
5,Hospital Receptionist,NaN,Remote,Thane,"₹ 10,000 - 12,000 /month"
6,Course Sales,NaN,Remote,Noida (...,"₹ 20,000 - 30,000 /month"
7,Academic Management,NaN,Remote,Delhi,"₹ 16,000 - 28,000 /month"
8,Sales,NaN,Remote,Pune,"₹ 10,000 - 15,000 /month"
9,Client Acquisition,NaN,Remote,"Pune, Fursungi, Autadwadi Handewadi, Katraj","₹ 9,000 - 11,001 /month"


In [48]:
# Data cleaning
df.isnull().sum() 

Title        1
Company     41
Location     0
Duration     1
Stipend      1
dtype: int64

In [49]:
df.columns=df.columns.str.strip()
df =df.dropna(subset=['Title'])

In [50]:
df['Company'] =df['Company'].fillna('Not listed')

In [51]:
df['Scraped_Date']=datetime.now()

In [52]:
df.head()

,Title,Company,Location,Duration,Stipend,Scraped_Date
0,Data Entry,Not listed,Remote,Gurgaon,"₹ 8,000 - 10,000 /month",2026-03-08 15:29:25.716232
1,Support Associate,Not listed,Remote,Work from home,"₹ 1,000 /month",2026-03-08 15:29:25.716232
2,Management - Presentation Specialist,Not listed,Remote,Gurgaon,"₹ 15,000 - 25,000 /month",2026-03-08 15:29:25.716232
3,Customer Service/Customer Support,Not listed,Remote,Work from home,"₹ 5,000 - 12,000 /month",2026-03-08 15:29:25.716232
4,Client Engagement & Wealth Management Support,Not listed,Remote,Mumbai,"₹ 3,000 - 7,000 /month",2026-03-08 15:29:25.716232


In [54]:
# Loading cleaned data into a SQLite database
# SQLite chosen for its lightweight nature — no server needed
conn = sqlite3.connect("internshala_jobs.db") #connecting

In [55]:
df.to_sql("jobs", conn, if_exists="replace", index=False)# Loading DataFrame into 'jobs' table

40

In [56]:
result = pd.read_sql("SELECT COUNT(*) FROM jobs", conn)
print("Total jobs:", result.iloc[0,0]) #checking

Total jobs: 40


In [ ]:
conn.close()


## Key Findings
- Most internships are **remote** positions
- Stipends range from **₹1,000 to ₹25,000/month**
- Majority of roles don't publicly list the company name